# 🏜️ Sécheresse — Drought (CLIMADA)

Exploration du module `Drought` de CLIMADA pour modéliser le risque de sécheresse à partir de l'indice SPEI.

**Aléa** : Sécheresse (DR)  
**Zone** : France / bassin méditerranéen  
**Données** : SPEI Global Drought Monitor / synthétique  
**Métriques** : SPEI (Standardized Precipitation-Evapotranspiration Index), durée, sévérité

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from climada.hazard import Hazard, Centroids
from climada.entity import Exposures, ImpactFuncSet, ImpactFunc
from climada.engine import Impact

try:
    from climada.hazard import Drought
    from climada.entity.impact_funcs.drought import ImpfDrought
    PETALS_OK = True
except ImportError:
    PETALS_OK = False
    print("⚠️ climada_petals non disponible — on utilisera des données synthétiques")

print(f"CLIMADA petals disponible : {PETALS_OK}")

## 1. Aléa — Sécheresse (SPEI)

Le **SPEI** (Standardized Precipitation-Evapotranspiration Index) combine précipitation et évapotranspiration pour quantifier les anomalies hydriques :

| SPEI | Classification |
|------|---------------|
| > 2.0 | Extrêmement humide |
| 1.5 à 2.0 | Très humide |
| -1.0 à 1.0 | Normal |
| -1.5 à -1.0 | Sécheresse modérée |
| -2.0 à -1.5 | Sécheresse sévère |
| < -2.0 | Sécheresse extrême |

CLIMADA utilise le SPEI-12 (échelle 12 mois) pour capturer les sécheresses hydrologiques.

In [ ]:
# --- CLIMADA Drought (nécessite données SPEI) ---
# from climada.hazard import Drought
# drought = Drought()
# drought.set_area(lat=(-10, 50), lon=(30, 70))  # Europe
# drought.set_threshold(threshold=-1.5)
# drought.set_intensity_def('spei')
# drought.setup()

# --- Données synthétiques : SPEI mensuel sur grille France ---
from scipy.sparse import csr_matrix, vstack

np.random.seed(42)

# Grille France élargie : 42-51°N, -5 à 10°E
lon = np.arange(-5, 10.5, 0.5)
lat = np.arange(42, 51.5, 0.5)
lon_grid, lat_grid = np.meshgrid(lon, lat)
lon_flat = lon_grid.flatten()
lat_flat = lat_grid.flatten()
n_centroids = len(lon_flat)

centroids = Centroids.from_lat_lon(lat_flat, lon_flat)
print(f"Centroids : {centroids.size} points sur la France")

# Générer 30 ans de SPEI-12 annuel (min annuel)
n_years = 30
years = np.arange(1993, 1993 + n_years)
frequency = np.ones(n_years) / n_years

# Tendance climatique : assèchement progressif dans le sud
lat_factor = (lat_flat - 51) / (42 - 51)  # 0 au nord, 1 au sud
trend = np.linspace(0, -0.5, n_years)  # tendance sèche

intensity_list = []
for y in range(n_years):
    # SPEI de base : bruit spatial corrélé
    base_spei = np.random.normal(0, 0.8, n_centroids)
    
    # Gradient nord-sud + tendance
    base_spei += lat_factor * (-0.3) + trend[y] * lat_factor
    
    # Années de sécheresse marquées (2003, 2018, 2022 analogues)
    if y in [10, 25, 29]:  # indices pour 2003, 2018, 2022
        base_spei -= np.random.uniform(0.5, 1.5, n_centroids) * lat_factor
    
    # On ne garde que les valeurs négatives (sécheresse)
    # L'intensité = -SPEI (plus c'est positif, plus c'est sec)
    drought_intensity = np.maximum(-base_spei, 0)
    drought_intensity[drought_intensity < 0.5] = 0  # seuil de sécheresse modérée
    
    intensity_list.append(csr_matrix(drought_intensity))

intensity_matrix = vstack(intensity_list)

drought_haz = Hazard(
    haz_type='DR',
    centroids=centroids,
    intensity=intensity_matrix,
    frequency=frequency,
    event_id=np.arange(1, n_years + 1),
    event_name=[f'Drought_{y}' for y in years],
    date=pd.date_range('1993-07-01', periods=n_years, freq='365D').to_numpy().astype('datetime64[ns]').astype(int) // 10**9,
    units='-SPEI'
)

print(f"Hazard : {drought_haz.size} années")
print(f"Intensité max (-SPEI) : {drought_haz.intensity.max():.2f}")
print(f"Cellules en sécheresse (total) : {drought_haz.intensity.nnz}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Trois années clés
key_years = [10, 25, 29]  # 2003, 2018, 2022
for idx, yi in enumerate(key_years):
    intensity = drought_haz.intensity[yi].toarray().flatten()
    sc = axes[idx].scatter(lon_flat, lat_flat, c=intensity, s=15, cmap='YlOrRd', vmin=0, vmax=3)
    axes[idx].set_title(f'Sécheresse {years[yi]} (-SPEI)')
    axes[idx].set_xlabel('Longitude')
    if idx == 0:
        axes[idx].set_ylabel('Latitude')
    plt.colorbar(sc, ax=axes[idx], label='-SPEI', shrink=0.8)

plt.suptitle('Années de sécheresse marquées', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Analyse temporelle et spatiale

Évolution de la sécheresse sur 30 ans : tendance, gradient nord-sud, et sévérité cumulée.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Sévérité moyenne par année
mean_intensity = np.array(drought_haz.intensity.mean(axis=1)).flatten()
axes[0, 0].bar(years, mean_intensity, color='sandybrown', alpha=0.8)
z = np.polyfit(years, mean_intensity, 1)
axes[0, 0].plot(years, np.polyval(z, years), 'r--', linewidth=2, 
                label=f'Tendance : {z[0]:+.4f}/an')
axes[0, 0].set_xlabel('Année')
axes[0, 0].set_ylabel('Sévérité moyenne (-SPEI)')
axes[0, 0].set_title('Sévérité moyenne annuelle')
axes[0, 0].legend()

# 2. Surface touchée (% cellules en sécheresse)
pct_affected = np.array((drought_haz.intensity > 0).sum(axis=1)).flatten() / n_centroids * 100
axes[0, 1].bar(years, pct_affected, color='peru', alpha=0.8)
axes[0, 1].set_xlabel('Année')
axes[0, 1].set_ylabel('% surface touchée')
axes[0, 1].set_title('Surface en sécheresse')

# 3. Gradient nord-sud
north_mask = lat_flat > 47
south_mask = lat_flat < 45
mean_north = np.array(drought_haz.intensity[:, north_mask].mean(axis=1)).flatten()
mean_south = np.array(drought_haz.intensity[:, south_mask].mean(axis=1)).flatten()
axes[1, 0].plot(years, mean_north, 'b-', linewidth=2, label='Nord (>47°N)')
axes[1, 0].plot(years, mean_south, 'r-', linewidth=2, label='Sud (<45°N)')
axes[1, 0].set_xlabel('Année')
axes[1, 0].set_ylabel('Sévérité (-SPEI)')
axes[1, 0].set_title('Gradient nord-sud')
axes[1, 0].legend()

# 4. Fréquence de sécheresse par cellule
drought_freq = np.array((drought_haz.intensity > 1.0).sum(axis=0)).flatten() / n_years * 100
sc = axes[1, 1].scatter(lon_flat, lat_flat, c=drought_freq, s=15, cmap='RdYlGn_r', vmin=0)
axes[1, 1].set_title('Fréquence sécheresse sévère (-SPEI > 1)')
axes[1, 1].set_xlabel('Longitude')
axes[1, 1].set_ylabel('Latitude')
plt.colorbar(sc, ax=axes[1, 1], label='% années', shrink=0.8)

plt.tight_layout()
plt.show()

## 3. Exposition — Actifs agricoles et urbains

Portefeuille mixte : zones agricoles (valeur de production) et zones urbaines (infrastructure d'approvisionnement en eau).

In [ ]:
from shapely.geometry import Point
import geopandas as gpd

n_assets = 350
np.random.seed(789)

# Mix urbain (30%) + agricole (70%)
cities_fr = {
    'Paris': (2.35, 48.86, 0.08),
    'Lyon': (4.83, 45.76, 0.05),
    'Marseille': (5.37, 43.30, 0.05),
    'Toulouse': (1.44, 43.60, 0.04),
    'Bordeaux': (-0.58, 44.84, 0.04),
    'Nantes': (-1.55, 47.22, 0.04),
}

# Zones agricoles
agri_zones = {
    'Beauce': (1.5, 48.0, 0.12),
    'Aquitaine': (0.0, 44.5, 0.10),
    'Languedoc': (3.5, 43.5, 0.10),
    'Bourgogne': (3.5, 47.0, 0.08),
    'Champagne': (3.5, 48.8, 0.08),
    'Provence': (5.5, 43.8, 0.08),
    'Val_Loire': (1.0, 47.5, 0.10),
}

lons, lats, values, types = [], [], [], []

# Urbain
for city, (clon, clat, weight) in cities_fr.items():
    n = int(n_assets * weight)
    lons.extend(np.random.normal(clon, 0.2, n))
    lats.extend(np.random.normal(clat, 0.15, n))
    values.extend(np.random.exponential(15e6, n))
    types.extend(['urban'] * n)

# Agricole
for zone, (clon, clat, weight) in agri_zones.items():
    n = int(n_assets * weight)
    lons.extend(np.random.normal(clon, 0.8, n))
    lats.extend(np.random.normal(clat, 0.5, n))
    values.extend(np.random.exponential(2e6, n))
    types.extend(['agriculture'] * n)

remaining = n_assets - len(lons)
if remaining > 0:
    lons.extend(np.random.uniform(-3, 8, remaining))
    lats.extend(np.random.uniform(43, 50, remaining))
    values.extend(np.random.exponential(3e6, remaining))
    types.extend(['agriculture'] * remaining)

gdf = gpd.GeoDataFrame({
    'value': np.array(values[:n_assets]),
    'impf_DR': np.ones(n_assets, dtype=int),
    'asset_type': types[:n_assets],
    'geometry': [Point(lo, la) for lo, la in zip(lons[:n_assets], lats[:n_assets])]
}, crs='EPSG:4326')

exposure = Exposures(gdf)
exposure.check()

print(f"Exposition : {len(exposure.gdf)} actifs")
print(f"  Urbain : {(exposure.gdf['asset_type'] == 'urban').sum()}")
print(f"  Agricole : {(exposure.gdf['asset_type'] == 'agriculture').sum()}")
print(f"Valeur totale : {exposure.gdf['value'].sum():,.0f} USD")

## 4. Vulnérabilité — Fonctions d'impact sécheresse

CLIMADA propose 3 types de fonctions pour la sécheresse :
- **`from_default()`** : seuil simple (step function)
- **`from_default_sum()`** : intégration de la sévérité cumulée
- **`from_step()`** : step function paramétrable

La difficulté de la sécheresse : c'est un aléa **lent** (onset progressif) dont l'impact dépend autant de la durée que de l'intensité.

In [ ]:
intensity_range = np.arange(0, 4, 0.01)

# Step function : pas de dommage sous seuil, linéaire au-dessus
threshold = 1.0
mdr_step = np.where(intensity_range > threshold, 
                     np.clip((intensity_range - threshold) / 2.0, 0, 0.8), 0)

# Sigmoïde progressive
mdr_sigmoid = 0.8 / (1 + np.exp(-3 * (intensity_range - 1.5)))
mdr_sigmoid[intensity_range < 0.5] = 0

# Linéaire simple
mdr_linear = np.clip(intensity_range / 3.0, 0, 0.6)
mdr_linear[intensity_range < 0.5] = 0

impf_step = ImpactFunc(
    id=1, haz_type='DR',
    intensity=intensity_range,
    mdd=mdr_step,
    paa=np.ones_like(mdr_step),
    intensity_unit='-SPEI',
    name='Step (seuil 1.0)'
)

impf_sigmoid = ImpactFunc(
    id=2, haz_type='DR',
    intensity=intensity_range,
    mdd=mdr_sigmoid,
    paa=np.ones_like(mdr_sigmoid),
    intensity_unit='-SPEI',
    name='Sigmoïde'
)

impf_linear = ImpactFunc(
    id=3, haz_type='DR',
    intensity=intensity_range,
    mdd=mdr_linear,
    paa=np.ones_like(mdr_linear),
    intensity_unit='-SPEI',
    name='Linéaire'
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(intensity_range, mdr_step, 'b-', linewidth=2, label='Step (seuil)')
ax.plot(intensity_range, mdr_sigmoid, 'r--', linewidth=2, label='Sigmoïde')
ax.plot(intensity_range, mdr_linear, 'g:', linewidth=2, label='Linéaire')
ax.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5, label='SPEI = -1 (modéré)')
ax.axvline(x=1.5, color='gray', linestyle='--', alpha=0.5, label='SPEI = -1.5 (sévère)')
ax.set_xlabel('Intensité (-SPEI)')
ax.set_ylabel('MDR')
ax.set_title('Fonctions de vulnérabilité — Sécheresse')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Impact — Calcul des pertes

In [ ]:
results = {}
for name, impf in [('Step', impf_step), ('Sigmoïde', impf_sigmoid), ('Linéaire', impf_linear)]:
    impf_use = ImpactFunc(
        id=1, haz_type='DR',
        intensity=impf.intensity, mdd=impf.mdd, paa=impf.paa,
        intensity_unit='-SPEI', name=name
    )
    impf_set = ImpactFuncSet([impf_use])
    
    imp = Impact()
    imp.calc(exposure, impf_set, drought_haz, save_mat=True)
    
    results[name] = {
        'eai': imp.aai_agg,
        'max_event': imp.at_event.max(),
        'freq_curve': imp.calc_freq_curve(),
        'impact': imp
    }
    print(f"{name:12s} — EAI: {imp.aai_agg:>12,.0f} USD | Max: {imp.at_event.max():>12,.0f} USD")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {'Step': 'steelblue', 'Sigmoïde': 'coral', 'Linéaire': 'green'}

# 1. Courbes de fréquence
for name, res in results.items():
    fc = res['freq_curve']
    axes[0].plot(fc.return_per, fc.impact / 1e6, linewidth=2, color=colors[name], label=name)
axes[0].set_xlabel('Période de retour (ans)')
axes[0].set_ylabel('Perte (M USD)')
axes[0].set_title('Courbes de fréquence')
axes[0].set_xscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Pertes temporelles (step function)
imp_step_result = results['Step']['impact']
axes[1].bar(years, imp_step_result.at_event / 1e6, color='sandybrown', alpha=0.8)
axes[1].set_xlabel('Année')
axes[1].set_ylabel('Perte (M USD)')
axes[1].set_title('Pertes annuelles (Step)')
axes[1].grid(True, alpha=0.3, axis='y')

# 3. EAI comparaison
eai_vals = [res['eai'] / 1e6 for res in results.values()]
bars = axes[2].bar(results.keys(), eai_vals, color=[colors[n] for n in results.keys()], alpha=0.8)
axes[2].set_ylabel('EAI (M USD/an)')
axes[2].set_title('Expected Annual Impact')
for bar, val in zip(bars, eai_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                 f'{val:.1f}M', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Synthèse

| Aspect | Détail |
|--------|--------|
| Source données | SPEI Global Drought Monitor (Vicente-Serrano et al.) |
| Variable | SPEI-12 (échelle 12 mois) |
| Résolution | 0.5° (~50 km) |
| Couverture | 1901-présent |
| Particularité | Aléa lent — impact fonction de durée ET intensité |

### Points clés
- La sécheresse est un aléa à **onset lent** : les fonctions d'impact doivent capturer la durée
- Le gradient nord-sud en France est marqué : le sud méditerranéen est 2-3× plus exposé
- La tendance climatique montre un assèchement progressif (cohérent avec les projections CMIP6)
- Le choix de la fonction d'impact (step vs sigmoïde) change drastiquement les résultats

### Prochaines étapes
- Intégrer les données SPEI réelles (CRU/GPCC)
- Coupler avec les projections CMIP6 (cf. notebook 04)
- Différencier les fonctions par secteur (agriculture vs urbain)
- Explorer le SPEI multi-échelles (1, 3, 6, 12, 24 mois)